# Lab 3: Neural Networks

For this lab you will need to install a few more packages: torch, torchvision and tensorflow.  
Tip: copy the environment from the previous lab into a new one, and then install the new packages.  
Avoid messing p with a working environment due to possible version compatibility issues.  

## General imports

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib as mp
import sklearn as sk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, classification_report

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch
import torchvision
import torchvision.transforms as T
from torchvision import models
from torchvision.io import read_image
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset
from torch.utils.data.sampler import SubsetRandomSampler

## Load dataset

Input Variables (X):
0. Number of times pregnant
1. Plasma glucose concentration at 2 hours in an oral glucose tolerance test
2. Diastolic blood pressure (mm Hg)
3. Triceps skin fold thickness (mm)
4. 2-hour serum insulin (mu U/ml)
5. Body mass index (weight in kg/(height in m)^2)
6. Diabetes pedigree function
7. Age (years)

Output Variables (y):
8. Class variable (0 or 1)

See: [source](https://machinelearningmastery.com/develop-your-first-neural-network-with-pytorch-step-by-step/)

In [ ]:
# load the dataset, split into input (X) and output (y) variables
dataset = np.loadtxt('pima-indians-diabetes.csv', delimiter=',')
dataset


In [ ]:
dataeset = pd.read_csv('pima-indians-diabetes.csv')
dataeset

In [ ]:
#plot class distribution
unique, counts = np.unique(dataeset['1'], return_counts=True)
plt.figure(figsize = (4,4))
ax = sns.barplot(x=unique,
                 y=counts).set_title('DataSet class distribution')
ticks = plt.xticks(rotation=45)

In [ ]:

X = dataset[:,0:8]
y = dataset[:,8]

## Data cleaning and preparation

In this notebook we won't go through all the data preparation steps again, but focus on the NN.  
Remember to use all the resources you learned during labs 1 and 2 to understand, clean and prepare your data.

## Pytorch example

Transform the data in the correct input shape

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

Split dataset

In [ ]:
#split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                              test_size=0.3, 
                                              random_state=42)

Define your model

In [ ]:
# option 2
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(8, 12)
        self.act1 = nn.ReLU()
        self.hidden2 = nn.Linear(12, 8)
        self.act2 = nn.ReLU()
        self.output = nn.Linear(8, 1)
        self.act_output = nn.Sigmoid()

    def forward(self, x):
        x = self.act1(self.hidden1(x))
        x = self.act2(self.hidden2(x))
        x = self.act_output(self.output(x))
        return x

model = Net()

In [ ]:
print(model)

In [ ]:
# option 1
model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

In [ ]:
print(model)

### Define loss function

Tip:
On previous labs we used resampling to deal with imbalanced datasets.  
Another different of dealing with this imbalance is assigning class weights on the model's loss function.  
Use the parameter 'weight' in the loss function to deal with the imbalanced dataset  
See sklearn.utils.class_weight.compute_class_weight

In [ ]:
# some examples
# criterion = nn.CrossEntropyLoss()
criterion = nn.BCELoss() # binary cross entropy
print (criterion)

DATA AUGMENTATION

In [ ]:
# from imblearn.over_sampling import RandomOverSampler, SMOTE

# # Perform random oversampling
# # ros = RandomOverSampler(random_state=0)
# # X_train_resamp, y_train_resamp = ros.fit_resample(X_train, y_train)
# X_train_resamp, y_train_resamp = SMOTE().fit_resample(X_train, y_train)

In [ ]:
# #plot class distribution
# unique, counts = np.unique(y_train_resamp, return_counts=True)
# plt.figure(figsize = (4,4))
# ax = sns.barplot(x=unique,
#                  y=counts).set_title('DataSet class distribution')
# ticks = plt.xticks(rotation=45)

In [ ]:
# X_train = torch.tensor(X_train_resamp, dtype=torch.float32)
# y_train = torch.tensor(y_train_resamp, dtype=torch.float32).reshape(-1, 1)

Define optimizer

In [ ]:
# some examples
# optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(optimizer)

Train the model

Below, we perform a loop over epochs, where for each training epoch the code:

- Gets a batch of training data
- Zeros the optimizer’s gradients
- Performs an inference - that is, gets predictions from the model for an input batch
- Calculates the loss for that set of predictions vs. the labels on the dataset
- Calculates the backward gradients over the learning weights
- Tells the optimizer to perform one learning step - that is, adjust the model’s learning weights based on the observed gradients for this batch, according to the optimization algorithm we chose
- Reports on the loss for every 10 batches.

See: [source](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html#the-training-loop)

In [ ]:
n_epochs = 500
batch_size = 20

training_losses = []
for epoch in range(n_epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for count, i in enumerate(range(0, len(X_train), batch_size)):
        
        # get the inputs
        inputs = X_train[i:i+batch_size]
        labels = y_train[i:i+batch_size]

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model(inputs)

        # Compute the loss and its gradients
        loss = criterion(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if count % 10 == 9:    # print every 10 mini-batches
            last_loss = running_loss / 10
            print(f'[{epoch + 1}, {count + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0

    training_losses.append(round(last_loss,3))

print('Finished Training')

In [ ]:
plt.plot(training_losses, label='Train loss')
plt.legend(frameon=False)
plt.show()

Save model

Tip: save the model with useful names that will help you identify it later on.

In [ ]:
PATH = './pytorch_model.pth'
torch.save(model.state_dict(), PATH)

Evaluate model

In [ ]:
# compute accuracy
with torch.no_grad():
    y_pred = model(X_test)

accuracy = (y_pred.round() == y_test).float().mean()
print(f"Accuracy {accuracy}")

In [ ]:
# make class predictions with the model
y_pred = (model(X_test) > 0.5).int() #binary classification
# print some predictions and true labels
for i in range(20):
    print(f'pred: {y_pred[i][0]} ; true {int(y_test[i])}')

Classification report

In [ ]:
print(sk.metrics.classification_report(y_test, y_pred))

Is our model overfitting or underfitting?

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [ ]:
# try with fewer epochs
n_epochs = 100

In [ ]:
for epoch in range(n_epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for count, i in enumerate(range(0, len(X_train), batch_size)):
        
        # get the inputs
        inputs = X_train[i:i+batch_size]
        labels = y_train[i:i+batch_size]

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model(inputs)

        # Compute the loss and its gradients
        loss = criterion(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

print('Finished Training')

In [ ]:
# compute accuracy
with torch.no_grad():
    y_pred = model(X_test)

accuracy = (y_pred.round() == y_test).float().mean()
print(f"Accuracy {accuracy}")

In [ ]:
# make class predictions with the model
y_pred = (model(X_test) > 0.5).int() #binary classification
# print some predictions and true labels
for i in range(20):
    print(f'pred: {y_pred[i][0]} ; true {int(y_test[i])}')

In [ ]:
print(sk.metrics.classification_report(y_test, y_pred))

Avoiding overfitting

Once per epoch perform validation by checking our relative loss on a set of data that was not used for training, and report this.  
This can be used to find a good number of epochs. Then, train a model on the full train dataset using this number of epochs.  
Here we use only one validation set.... what problems could this cause and how would you solve them?

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [ ]:

def train_one_epoch(X_train, y_train):

    running_loss = 0.0
    last_loss = 0.0
    for count, i in enumerate(range(0, len(X_train), batch_size)):
        
        # get the inputs
        inputs = X_train[i:i+batch_size]
        labels = y_train[i:i+batch_size]

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model(inputs)

        # Compute the loss and its gradients
        loss = criterion(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if count % 10 == 9:    # print every 10 mini-batches
            last_loss = running_loss / 10
            print(f'[{epoch + 1}, {count + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0
    
    return last_loss

In [ ]:
best_val_loss = 1_000_000.
n_epochs = 300
training_losses = []
val_losses = []

train_input, val_input, train_label, val_label = train_test_split(X_train, y_train,
                                            test_size=0.2, 
                                            random_state=42,
                                            )

for epoch in range(n_epochs):  # loop over the dataset multiple times

    # Make sure gradient tracking is on, and do a pass over the data
    model.train(True)

    avg_loss = train_one_epoch(train_input, train_label)

    running_val_loss = 0.0
    # Set the model to evaluation mode, disabling dropout and using population
    # statistics for batch normalization.
    model.eval()

    # Disable gradient computation and reduce memory consumption.
    val_output = model(val_input)
    val_loss = criterion(val_output, val_label)
    running_val_loss += val_loss

    avg_val_loss = running_val_loss / (i + 1)
    print('LOSS train {} val {}'.format(avg_loss, avg_val_loss))

    # Track parameters
    training_losses.append(avg_loss)
    val_losses.append(avg_val_loss.detach().numpy())

    # Track best performance, and save the model's state
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model = {
            'epoch': epoch,
            'val_loss':best_val_loss
        }


In [ ]:
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

ax1.plot(training_losses, label='Training loss')
ax2.plot(val_losses, label='Validation loss', color='orange')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.show()

AUC-ROC

Get Probabilities from the Model

In [ ]:
model.eval()

with torch.no_grad():
    y_scores = model(X_test).cpu().numpy().ravel()   # probabilities
    y_true = y_test.cpu().numpy().ravel()

Compute

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
auc_score = roc_auc_score(y_true, y_scores)

print(f"ROC AUC: {auc_score:.4f}")

Plotting

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

Image classification example

See: [reference](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html).